In [4]:
#Import packages and set up Neo4
from dotenv import load_dotenv
import os

from langchain_community.graphs import Neo4jGraph

# Warning control
import warnings
warnings.filterwarnings("ignore")
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')
kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

In [5]:
#Querying the movie knowledge graph
cypher = """
  MATCH (n) 
  RETURN count(n)
  """
result = kg.query(cypher)
result

[{'count(n)': 171}]

In [6]:
cypher = """
  MATCH (n) 
  RETURN count(n) AS numberOfNodes
  """
result = kg.query(cypher)
result


[{'numberOfNodes': 171}]

In [7]:
print(f"There are {result[0]['numberOfNodes']} nodes in this graph.")

There are 171 nodes in this graph.


In [8]:
#Match only the Movie nodes by specifying the node label
cypher = """
  MATCH (n:Movie) 
  RETURN count(n) AS numberOfMovies
  """
kg.query(cypher)

[{'numberOfMovies': 38}]

In [9]:
#Change the variable name in the node pattern match for improved readability
cypher = """
  MATCH (m:Movie) 
  RETURN count(m) AS numberOfMovies
  """
kg.query(cypher)

[{'numberOfMovies': 38}]

In [12]:
#Match only the Person nodes
cypher = """
  MATCH (people:Person) 
  RETURN count(people) AS numberOfPeople
  """
result=kg.query(cypher)
print(result)

#Match a single person by specifying the value of the name property on the Person node
cypher = """
  MATCH (tom:Person {name:"Tom Hanks"}) 
  RETURN tom
  """
result=kg.query(cypher)
print(result)

#Match a single Movie by specifying the value of the title property
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas
  """
result = kg.query(cypher)
print(result)

[{'numberOfPeople': 133}]
[{'tom': {'born': 1956, 'name': 'Tom Hanks'}}]
[{'cloudAtlas': {'tagline': 'Everything is connected', 'title': 'Cloud Atlas', 'released': 2012}}]


In [13]:
#Return only the released property of the matched Movie node
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas.released
  """
result = kg.query(cypher)
print(result)

#Return two properties
cypher = """
  MATCH (cloudAtlas:Movie {title:"Cloud Atlas"}) 
  RETURN cloudAtlas.released, cloudAtlas.tagline
  """
result = kg.query(cypher)
print(result)

[{'cloudAtlas.released': 2012}]
[{'cloudAtlas.released': 2012, 'cloudAtlas.tagline': 'Everything is connected'}]


In [14]:
#Cypher patterns with conditional matching
cypher = """
  MATCH (nineties:Movie) 
  WHERE nineties.released >= 1990 
    AND nineties.released < 2000 
  RETURN nineties.title
  """
result = kg.query(cypher)
print(result)

[{'nineties.title': 'The Matrix'}, {'nineties.title': "The Devil's Advocate"}, {'nineties.title': 'A Few Good Men'}, {'nineties.title': 'As Good as It Gets'}, {'nineties.title': 'What Dreams May Come'}, {'nineties.title': 'Snow Falling on Cedars'}, {'nineties.title': "You've Got Mail"}, {'nineties.title': 'Sleepless in Seattle'}, {'nineties.title': 'Joe Versus the Volcano'}, {'nineties.title': 'When Harry Met Sally'}, {'nineties.title': 'That Thing You Do'}, {'nineties.title': 'The Birdcage'}, {'nineties.title': 'Unforgiven'}, {'nineties.title': 'Johnny Mnemonic'}, {'nineties.title': 'The Green Mile'}, {'nineties.title': 'Hoffa'}, {'nineties.title': 'Apollo 13'}, {'nineties.title': 'Twister'}, {'nineties.title': 'Bicentennial Man'}, {'nineties.title': 'A League of Their Own'}]


In [15]:
#Pattern matching with multiple nodes
cypher = """
  MATCH (actor:Person)-[:ACTED_IN]->(movie:Movie) 
  RETURN actor.name, movie.title LIMIT 10
  """
result = kg.query(cypher)
print(result)

cypher = """
  MATCH (tom:Person {name: "Tom Hanks"})-[:ACTED_IN]->(tomHanksMovies:Movie) 
  RETURN tom.name,tomHanksMovies.title
  """
result = kg.query(cypher)
print(result)

cypher = """
  MATCH (tom:Person {name:"Tom Hanks"})-[:ACTED_IN]->(m)<-[:ACTED_IN]-(coActors) 
  RETURN coActors.name, m.title
  """
result = kg.query(cypher)
print(result)

[{'actor.name': 'Emil Eifrem', 'movie.title': 'The Matrix'}, {'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix'}, {'actor.name': 'Laurence Fishburne', 'movie.title': 'The Matrix'}, {'actor.name': 'Carrie-Anne Moss', 'movie.title': 'The Matrix'}, {'actor.name': 'Keanu Reeves', 'movie.title': 'The Matrix'}, {'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix Reloaded'}, {'actor.name': 'Laurence Fishburne', 'movie.title': 'The Matrix Reloaded'}, {'actor.name': 'Carrie-Anne Moss', 'movie.title': 'The Matrix Reloaded'}, {'actor.name': 'Keanu Reeves', 'movie.title': 'The Matrix Reloaded'}, {'actor.name': 'Hugo Weaving', 'movie.title': 'The Matrix Revolutions'}]
[{'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'Apollo 13'}, {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': "You've Got Mail"}, {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'A League of Their Own'}, {'tom.name': 'Tom Hanks', 'tomHanksMovies.title': 'Joe Versus the Volcano'}, {'tom.name': 'Tom Hanks', 'tomHank

In [16]:
#Delete data from the graph
cypher = """
MATCH (emil:Person {name:"Emil Eifrem"})-[actedIn:ACTED_IN]->(movie:Movie)
RETURN emil.name, movie.title
"""
result = kg.query(cypher)
print(result)

cypher = """
MATCH (emil:Person {name:"Emil Eifrem"})-[actedIn:ACTED_IN]->(movie:Movie)
DELETE actedIn
"""
result = kg.query(cypher)
print(result)

[{'emil.name': 'Emil Eifrem', 'movie.title': 'The Matrix'}]
[]
